## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch

from mingpt.model import CausalSelfAttention, GPTConfig, GPT1Config
from mingpt.char_dataset import CharDataset

In [5]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

## 01 `CausalSelfAttention` (1)

In [28]:
config = GPT1Config(vocab_size=65, block_size=64)
config.n_head = 4
config.n_embd = 128
head_size = config.n_embd // config.n_head

torch.manual_seed(0)
dummy_x = torch.randn(2, config.block_size, config.n_embd, device=device)
print(f"dummy_x shape: {dummy_x.shape}")  # (2, block_size, n_embd)
print(f"dummy_x first 5 elements: {dummy_x[0, 0, :5]}")

torch.manual_seed(0)
attn = CausalSelfAttention(config)
out_attn = attn(dummy_x)
print(f"attn output shape: {out_attn.shape}")  # (2, block_size, n_embd)
print(f"attn output first 5 elements: {out_attn[0, 0, :5]}")

dummy_x shape: torch.Size([2, 64, 128])
dummy_x first 5 elements: tensor([-1.1258, -1.1524, -0.2506, -0.4339,  0.8487])
attn output shape: torch.Size([2, 64, 128])
attn output first 5 elements: tensor([-0.1890, -0.3121, -0.0661,  0.1308, -0.6897], grad_fn=<SliceBackward0>)


## 02 `CausalSelfAttention` (2)

In [27]:
# 1. Config matching gpt.py
config = GPTConfig(
    vocab_size=65,
    block_size=64,
    n_embd=128,
    n_head=4,
    attn_pdrop=0.0,
    resid_pdrop=0.0
)

torch.manual_seed(0)
csa = CausalSelfAttention(config).to('cpu')
csa.eval()

# 2. Save its state_dict to a file
torch.save(csa.state_dict(), 'csa_weights.pt')
print("Saved csa_weights.pt successfully!")

Saved csa_weights.pt successfully!


Let's disable dropout for debugging purposes.

In [30]:
# 1. Config matching gpt.py
config = GPTConfig(
    vocab_size=65,
    block_size=64,
    n_embd=128,
    n_head=4,
    attn_pdrop=0.0,
    resid_pdrop=0.0
)

print(f"===Testing CausalSelfAttention===")

torch.manual_seed(0)
dummy_x = torch.randn(2, config.block_size, config.n_embd, device=device)
print(f"dummy_x shape: {dummy_x.shape}")  # (2, block_size, n_embd)
print(f"dummy_x first 5 elements: {dummy_x[0, 0, :5]}")

torch.manual_seed(0)
csa = CausalSelfAttention(config).to('cpu')
csa.eval()

out_attn = csa(dummy_x)
print(f"attn output first 5 elements: {out_attn[0, 0, :5]}")

# 2. Save its state_dict to a file
torch.save(csa.state_dict(), 'csa_weights.pt')
print("Saved csa_weights.pt successfully!")

===Testing CausalSelfAttention===
dummy_x shape: torch.Size([2, 64, 128])
dummy_x first 5 elements: tensor([-1.1258, -1.1524, -0.2506, -0.4339,  0.8487])
attn output first 5 elements: tensor([-0.1582, -0.2532, -0.0594,  0.1068, -0.5539], grad_fn=<SliceBackward0>)
Saved csa_weights.pt successfully!


## 03 `CharDataset` class

In [13]:
char_data = CharDataset()

Read 1,115,394 characters from input.txt
Set block size to 128
Vocabulary size: 65


In [14]:
x, y = char_data[0]

Chunk at index 0: 129 characters


In [12]:
len(x), len(y)

(128, 128)

In [15]:
x[:5], y[:5]

(tensor([18, 47, 56, 57, 58]), tensor([47, 56, 57, 58,  1]))

In [16]:
x[-5:], y[-5:]

(tensor([56,  1, 58, 53,  1]), tensor([ 1, 58, 53,  1, 42]))